# Classifier intent tahap 1 (GroUp)

Fine-tuning dua kandidat (`indobenchmark/indobert-lite-base-p1`, `intfloat/multilingual-e5-small`) pada `train.jsonl`, memilih yang terbaik berdasarkan macro-F1 di `val.jsonl`, lalu ekspor ONNX + kuantisasi int8 dinamis untuk transformers.js.

- Jalankan di Colab dengan GPU (Runtime > Change runtime type > GPU).
- Unggah `ai/training/requirements.txt`, `ai/data/train.jsonl`, dan `ai/data/val.jsonl` ke folder kerja Colab.
- **Set test tidak dimuat di notebook ini.** Angka test dihitung oleh `npm run eval` di repo.
- Hasil: `intent-classifier.zip`. Ekstrak ke `models/intent-classifier/` di repo, lalu jalankan `npm run model:calibrate`.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json, random, shutil, time
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, BertTokenizerFast, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments)

# Urutan harus sama persis dengan INTENTS di lib/agent/types.ts.
INTENTS = ["pesan_baru", "ubah_pesanan", "batal_pesanan", "cek_pesanan", "tanya_info",
           "tanya_harga_pasar", "catat_minat", "sapaan", "di_luar_topik"]
LABEL2ID = {label: i for i, label in enumerate(INTENTS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
CANDIDATES = ["indobenchmark/indobert-lite-base-p1", "intfloat/multilingual-e5-small"]
# IndoBERT-lite memakai config ALBERT tetapi kosakata WordPiece (vocab.txt), jadi AutoTokenizer salah memilih
# AlbertTokenizer (sentencepiece). Model card-nya memakai BertTokenizer.
TOKENIZER_CLASS = {"indobenchmark/indobert-lite-base-p1": BertTokenizerFast}


def load_tokenizer(model_id):
    return TOKENIZER_CLASS.get(model_id, AutoTokenizer).from_pretrained(model_id)
SEED, MAX_LENGTH, LR, BATCH, EPOCHS = 42, 64, 3e-5, 32, 6
print(torch.__version__, "GPU:", torch.cuda.is_available())

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

if not (Path("train.jsonl").exists() and Path("val.jsonl").exists()):
    from google.colab import files
    files.upload()

train_rows, val_rows = read_jsonl("train.jsonl"), read_jsonl("val.jsonl")
assert all(r["intent"] in LABEL2ID for r in train_rows + val_rows), "intent tak dikenal"
print(len(train_rows), "train,", len(val_rows), "val")

In [ ]:
class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, rows, tokenizer):
        self.encodings = tokenizer([r["text"] for r in rows], truncation=True, max_length=MAX_LENGTH)
        self.labels = [LABEL2ID[r["intent"]] for r in rows]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        item["labels"] = self.labels[i]
        return item


def compute_metrics(pred):
    y_pred = pred.predictions.argmax(-1)
    return {"accuracy": accuracy_score(pred.label_ids, y_pred),
            "macro_f1": f1_score(pred.label_ids, y_pred, average="macro", labels=list(range(len(INTENTS))), zero_division=0)}


def train(model_id):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    tokenizer = load_tokenizer(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=len(INTENTS), id2label=ID2LABEL, label2id=LABEL2ID)
    run_dir = Path("runs") / model_id.replace("/", "__")
    args = TrainingArguments(output_dir=str(run_dir), learning_rate=LR, per_device_train_batch_size=BATCH,
                             per_device_eval_batch_size=BATCH, num_train_epochs=EPOCHS, eval_strategy="epoch",
                             save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="macro_f1",
                             save_total_limit=1, seed=SEED, report_to="none", fp16=torch.cuda.is_available())
    trainer = Trainer(model=model, args=args, train_dataset=IntentDataset(train_rows, tokenizer),
                      eval_dataset=IntentDataset(val_rows, tokenizer), data_collator=DataCollatorWithPadding(tokenizer),
                      compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    started = time.time()
    trainer.train()
    metrics = trainer.evaluate()
    best_dir = Path("best") / model_id.replace("/", "__")
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))
    return {"model": model_id, "val_macro_f1": metrics["eval_macro_f1"], "val_accuracy": metrics["eval_accuracy"],
            "epochs": metrics.get("epoch"), "minutes": (time.time() - started) / 60, "path": str(best_dir)}


results = [train(model_id) for model_id in CANDIDATES]

In [ ]:
for r in sorted(results, key=lambda r: -r["val_macro_f1"]):
    print(f"{r['model']:40s} macro-F1 {r['val_macro_f1']:.4f}  akurasi {r['val_accuracy']:.4f}  epoch {r['epochs']}  {r['minutes']:.1f} menit")
best = max(results, key=lambda r: r["val_macro_f1"])
print("Dipilih:", best["model"])
summary = {"candidates": results, "chosen": best["model"],
           "hyperparameters": {"max_length": MAX_LENGTH, "lr": LR, "batch": BATCH, "max_epochs": EPOCHS, "seed": SEED},
           "data": {"train": len(train_rows), "val": len(val_rows)}}
Path("training_summary.json").write_text(json.dumps(summary, indent=2))

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

export_dir, final_dir = Path("export"), Path("intent-classifier")
shutil.rmtree(export_dir, ignore_errors=True)
shutil.rmtree(final_dir, ignore_errors=True)
ort_model = ORTModelForSequenceClassification.from_pretrained(best["path"], export=True)
ort_model.save_pretrained(export_dir)
AutoTokenizer.from_pretrained(best["path"], use_fast=True).save_pretrained(export_dir)

quantizer = ORTQuantizer.from_pretrained(export_dir)
quantizer.quantize(save_dir=export_dir / "q8",
                   quantization_config=AutoQuantizationConfig.avx2(is_static=False, per_channel=False))

(final_dir / "onnx").mkdir(parents=True)
shutil.copy(export_dir / "q8" / "model_quantized.onnx", final_dir / "onnx" / "model_quantized.onnx")
for name in ["config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]:
    if (export_dir / name).exists():
        shutil.copy(export_dir / name, final_dir / name)
shutil.copy("training_summary.json", final_dir / "training_summary.json")

config = json.loads((final_dir / "config.json").read_text())
assert [config["id2label"][str(i)] for i in range(len(INTENTS))] == INTENTS, "id2label tidak sama dengan INTENTS"
assert (final_dir / "tokenizer.json").exists(), "tokenizer.json wajib untuk transformers.js"

In [ ]:
import onnxruntime as ort

size_mb = (final_dir / "onnx" / "model_quantized.onnx").stat().st_size / 1e6
print(f"model_quantized.onnx: {size_mb:.1f} MB")
assert size_mb < 30, "melebihi batas 30 MB (NFR-7)"

session = ort.InferenceSession(str(final_dir / "onnx" / "model_quantized.onnx"))
tok = AutoTokenizer.from_pretrained(final_dir)
for text in ["mau ikut cabe 5 kilo", "kapan bayarnya kak", "halo kak", "bikinin puisi dong"]:
    enc = tok(text, return_tensors="np")
    feeds = {i.name: enc[i.name] for i in session.get_inputs() if i.name in enc}
    logits = session.run(None, feeds)[0][0]
    print(f"{text!r:28s} -> {INTENTS[int(logits.argmax())]}")

shutil.make_archive("intent-classifier", "zip", final_dir)
try:
    from google.colab import files
    files.download("intent-classifier.zip")
except ImportError:
    print("Zip:", Path("intent-classifier.zip").resolve())